# DEAD SLOT DIRECTOR — Kaggle / Wan 2.2 TI2V-5B

**Goal:** one-click-ish image-to-video with **no API keys, no .env, no paid service credentials**.

### Josh only has to do this:
1. Open this notebook in Kaggle.
2. Turn **GPU** on in Notebook settings.
3. Upload the approved 9:16 master frame into `/kaggle/working/input.png`.
4. Run all cells.
5. Download `/kaggle/working/dead_slot_shot.mp4`.

This notebook uses the public `Wan-AI/Wan2.2-TI2V-5B` weights and does not require a Hugging Face token.


In [ ]:
import os, subprocess, sys, textwrap, torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Kaggle: Settings → Accelerator → GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
%%capture
!git clone -q https://github.com/Wan-Video/Wan2.2.git /kaggle/working/Wan2.2 || true
%cd /kaggle/working/Wan2.2
!pip -q install -r requirements.txt
!pip -q install 'huggingface_hub[cli]'


In [ ]:
from pathlib import Path
model_dir = Path('/kaggle/working/Wan2.2-TI2V-5B')
if not model_dir.exists() or not any(model_dir.iterdir()):
    !huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir /kaggle/working/Wan2.2-TI2V-5B
else:
    print('Model already present.')


## Upload the approved frame
In Kaggle's file pane, upload the image and name it `input.png` in `/kaggle/working/`.

For **Shot 1**, keep the prompt restrained. We want real temporal generation, not a fake camera move.


In [ ]:
from pathlib import Path
INPUT = Path('/kaggle/working/input.png')
assert INPUT.exists(), 'Upload your approved frame as /kaggle/working/input.png first.'

PROMPT = '''Locked-off cinematic shot of the exact same empty tattoo studio at night. Preserve the chair, furniture, room layout, framing, perspective and overall lighting from the source image. The camera remains completely stationary. Natural temporal motion only: rain visibly crawls down the exterior glass, distant wet-street headlights pass softly outside and their reflections change naturally on the glass and floor, faint atmospheric haze shifts almost imperceptibly, and the fluorescent lighting has one subtle realistic ballast fluctuation. Nothing inside the room moves. No people enter. No object morphing. No camera pan, zoom, dolly, tilt or shake. Moody late-night realism, restrained, photoreal, cinematic.'''

print(PROMPT)


In [ ]:
%cd /kaggle/working/Wan2.2

# Wan 2.2 TI2V supports vertical 704x1280. Offload/T5-on-CPU are enabled to reduce VRAM pressure.
!python generate.py \
  --task ti2v-5B \
  --size 704*1280 \
  --ckpt_dir /kaggle/working/Wan2.2-TI2V-5B \
  --offload_model True \
  --convert_model_dtype \
  --t5_cpu \
  --image /kaggle/working/input.png \
  --prompt "$PROMPT"


In [ ]:
from pathlib import Path
import shutil, glob, os
candidates = list(Path('/kaggle/working/Wan2.2').rglob('*.mp4'))
if not candidates:
    raise FileNotFoundError('Wan finished without an MP4 I could find. Inspect the previous cell output.')
latest = max(candidates, key=lambda p: p.stat().st_mtime)
target = Path('/kaggle/working/dead_slot_shot.mp4')
shutil.copy2(latest, target)
print('READY:', target)
print('Source:', latest)


In [ ]:
from IPython.display import Video, display
display(Video('/kaggle/working/dead_slot_shot.mp4', embed=True))
